# Higiene de demanda

Qué aplica `prepare_daily_demand` (usado por `train` / `predict`):

1. Filtro no-producto (códigos basura + frases concretas; no tumba productos con “check” en el nombre)
2. Relleno de días sin venta a 0 (calendario continuo por SKU)
3. Winsorización al percentil 99 y exclusión del outlier one-shot `23843`

In [ ]:
from pathlib import Path
import sys

import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "inventario_ecommerce").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from inventario_ecommerce import config
from inventario_ecommerce.dataset import load_transactions, save_processed
from inventario_ecommerce.features import (
    build_daily_sku_demand,
    build_rolling_features,
    clean_transactions,
    compute_abc_classification,
    prepare_daily_demand,
    sales_by_product_last_quarter,
)
from inventario_ecommerce.modeling.train import temporal_backtest_baseline
from inventario_ecommerce.modeling.predict import forecast_30d_baseline, build_reorder_policy

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
raw = load_transactions()
clean = clean_transactions(raw)

sparse = build_daily_sku_demand(clean)
dense = prepare_daily_demand(clean)

print(f"Días con venta (sparse): {len(sparse):,}")
print(f"Días calendario (dense):  {len(dense):,}")
print(f"SKUs dense: {dense[config.COL_STOCK_CODE].nunique():,}")
print(f"Outliers excluidos: {sorted(config.OUTLIER_STOCK_CODES)}")
print(f"Cap winsor P{config.DAILY_QTY_WINSOR_PERCENTILE:.0%}: "
      f"{dense.loc[dense['QuantitySold'] > 0, 'QuantitySold'].max():,.1f}")

In [ ]:
abc = compute_abc_classification(sales_by_product_last_quarter(clean))
rolling = build_rolling_features(dense)
latest = (
    rolling.sort_values("Date")
    .groupby([config.COL_STOCK_CODE, config.COL_DESCRIPTION], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

sku_metrics, global_metrics = temporal_backtest_baseline(dense)
global_metrics

In [ ]:
forecast = forecast_30d_baseline(dense)
policy = build_reorder_policy(latest, forecast, abc)
print("Top 10 sin one-shots extremos:")
policy.head(10)

In [ ]:
save_processed(abc, "abc_last_quarter.csv")
save_processed(latest, "sku_rolling_features_latest.csv")
save_processed(sku_metrics, "forecast_backtest_by_sku.csv")
save_processed(global_metrics, "forecast_backtest_global.csv")
save_processed(policy, "inventory_reorder_recommendations.csv")
print("Artefactos guardados en data/processed/")

## Métrica

El MAE del backtest se calcula sobre el calendario completo (incluye días a 0). Es la lectura adecuada para demanda intermitente retail.